# Topology, Density, and C--C Topology Results

Loads CREAM results for full-DAG topology, full-DAG density, and C--C fixed-count topology experiments. Each section includes original CREAM as a reference row/curve when available.


In [ ]:

from pathlib import Path
import re
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').exists() and (PROJECT_ROOT.parent / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPORT_DIR = PROJECT_ROOT / 'notebook' / 'topology_density_c2c_exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'celeba': {
        'display': 'CelebA',
        'model': 'Standard_CelebA',
        'original_experiment': 'CREAM_best_celeba',
        'base_dag': PROJECT_ROOT / 'data/CelebA/final_DAG_unfair.csv',
        'data_root': PROJECT_ROOT / 'data/CelebA',
        'num_classes': 1,
    },
    'cub': {
        'display': 'CUB',
        'model': 'Standard_CUB',
        'original_experiment': 'CREAM_cub/seed_0',
        'base_dag': PROJECT_ROOT / 'data/CUB/CUB_DAG_only_Gc.csv',
        'data_root': PROJECT_ROOT / 'data/CUB',
        'num_classes': 200,
    },
    'cfmnist': {
        'display': 'Complete_Concept_FMNIST',
        'model': 'Standard_FashionMNIST',
        'original_experiment': 'CREAM_best_cfmnist',
        'base_dag': PROJECT_ROOT / 'data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv',
        'data_root': PROJECT_ROOT / 'data/FashionMNIST',
        'num_classes': 10,
    },
}

FAMILIES = {
    'topology_random_full': 'Full-DAG topology random, fixed full-DAG edge count',
    'density_random_full': 'Full-DAG density random, varied full-DAG edge count',
    'concept_concept_fixcount_topology': 'C--C-only topology replacement, fixed C--C edge count',
}

KEY_METRICS = [
    'dataset', 'family', 'variant', 'is_original_reference',
    'edge_count', 'original_edge_count', 'hamming_distance', 'graph_distance',
    'preserved_edges', 'edge_overlap', 'rewire_fraction', 'rewire_percent', 'graph_scope',
    'test_task_accuracy', 'test_concept_accuracy', 'concept_leakage', 'data_leakage',
    'intervention_auc_false', 'intervention_auc_true',
    'intervention_final_accuracy_false', 'intervention_final_accuracy_true',
    'delta_pred_task_to_original', 'delta_pred_concept_to_original',
    'delta_int_auc_false_to_original', 'delta_int_auc_true_to_original', 'delta_int_to_original',
    'CCI', 'PFI_concept_importance', 'PFI_side_importance',
    'c2y_baseline_accuracy', 'test_dropout_task_accuracy',
    'metrics_path',
]


def experiment_root(dataset, family):
    meta = DATASETS[dataset]
    return (
        PROJECT_ROOT / 'experiments' / meta['display'] / 'train_cbm' / meta['model'] /
        f'{dataset}_{family}' / f'CREAM_{dataset}_{family}'
    )


def original_root(dataset):
    meta = DATASETS[dataset]
    return (
        PROJECT_ROOT / 'experiments' / meta['display'] / 'train_cbm' / meta['model'] /
        'CREAM' / meta['original_experiment']
    )


def read_metric_row(csv_path):
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'Skipping unreadable metrics CSV {csv_path}: {exc}')
        return None
    if df.empty:
        return None
    return df.iloc[0].to_dict()


def choose_metric_csv(root):
    candidates = sorted((root / 'last_metrics').glob('*.csv'))
    candidates += sorted(root.glob('**/last_metrics/*.csv'))
    # Do not scan arbitrary root CSVs here; intermediate activation CSVs can be large.
    seen = set()
    deduped = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped[-1] if deduped else None


def load_result_metrics():
    rows = []
    for dataset in DATASETS:
        # Original CREAM reference once per family, so every section can display it.
        o_root = original_root(dataset)
        o_csv = choose_metric_csv(o_root)
        if o_csv is not None:
            row = read_metric_row(o_csv)
            if row is not None:
                for family in FAMILIES:
                    copied = dict(row)
                    copied.update({
                        'dataset': dataset,
                        'family': family,
                        'variant': 'original_CREAM',
                        'is_original_reference': True,
                        'metrics_path': str(o_csv),
                    })
                    rows.append(copied)
        else:
            print(f'Original CREAM metrics not found for {dataset}: {o_root}')

        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                print(f'Result root not found yet for {dataset}/{family}: {root}')
                continue
            for variant_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
                csv_path = choose_metric_csv(variant_dir)
                if csv_path is None:
                    continue
                row = read_metric_row(csv_path)
                if row is None:
                    continue
                row.update({
                    'dataset': dataset,
                    'family': family,
                    'variant': variant_dir.name,
                    'is_original_reference': False,
                    'metrics_path': str(csv_path),
                })
                rows.append(row)
    return pd.DataFrame(rows)


def intervention_accuracy_column(df):
    for col in ['test_task_accuracy', 'task_accuracy', 'accuracy', 'test_accuracy', 'test_acc']:
        if col in df.columns:
            return col
    matches = [col for col in df.columns if 'accuracy' in col.lower() or col.lower().endswith('_acc')]
    return matches[0] if matches else None


def choose_intervention_csv(root):
    candidates = sorted(root.glob('lightning_logs/**/intervention_results.csv'))
    candidates += sorted(root.glob('**/intervention_results.csv'))
    seen = set()
    deduped = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped[-1] if deduped else None


def load_result_interventions():
    rows = []
    for dataset in DATASETS:
        o_root = original_root(dataset)
        o_csv = choose_intervention_csv(o_root)
        if o_csv is not None:
            df = pd.read_csv(o_csv)
            if not df.empty:
                for family in FAMILIES:
                    copied = df.copy()
                    copied['dataset'] = dataset
                    copied['family'] = family
                    copied['variant'] = 'original_CREAM'
                    copied['is_original_reference'] = True
                    copied['intervention_path'] = str(o_csv)
                    rows.append(copied)
        else:
            print(f'Original CREAM intervention curve not found for {dataset}: {o_root}')

        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                continue
            for variant_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
                csv_path = choose_intervention_csv(variant_dir)
                if csv_path is None:
                    continue
                df = pd.read_csv(csv_path)
                if df.empty:
                    continue
                df['dataset'] = dataset
                df['family'] = family
                df['variant'] = variant_dir.name
                df['is_original_reference'] = False
                df['intervention_path'] = str(csv_path)
                rows.append(df)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def available(columns, requested):
    return [col for col in requested if col in columns]



def read_bool_dag(path):
    with open(path, newline='') as f:
        rows = list(csv.reader(f))
    matrix = np.array([[cell == 'True' for cell in row[1:]] for row in rows[1:]], dtype=bool)
    return matrix


def dag_variant_path(dataset, family, variant):
    meta = DATASETS[dataset]
    if variant == 'original_CREAM':
        return meta['base_dag']
    root = meta['data_root'] / family
    if not root.exists():
        return None

    patterns = []
    if family == 'density_random_full':
        patterns = [f'*{variant}.csv']
    elif family == 'topology_random_full':
        patterns = [f'*{variant}*.csv']
    elif family == 'concept_concept_fixcount_topology':
        patterns = [f'*{variant}.csv']
    else:
        patterns = [f'*{variant}*.csv']

    matches = []
    for pattern in patterns:
        matches.extend(sorted(root.glob(pattern)))
    return matches[0] if matches else None


def graph_scope_matrices(dataset, family, adj, original):
    if family == 'concept_concept_fixcount_topology':
        num_classes = DATASETS[dataset]['num_classes']
        num_concepts = adj.shape[0] - num_classes
        return adj[:num_concepts, :num_concepts], original[:num_concepts, :num_concepts], 'concept_concept'
    return adj, original, 'full_dag'


def graph_property_rows_for_results(metrics_df):
    if metrics_df.empty:
        return pd.DataFrame()
    rows = []
    original_cache = {}
    for _, metric_row in metrics_df[['dataset', 'family', 'variant']].drop_duplicates().iterrows():
        dataset = metric_row['dataset']
        family = metric_row['family']
        variant = metric_row['variant']
        meta = DATASETS[dataset]
        if dataset not in original_cache:
            original_cache[dataset] = read_bool_dag(meta['base_dag'])
        original_full = original_cache[dataset]
        dag_path = dag_variant_path(dataset, family, variant)
        if dag_path is None or not Path(dag_path).exists():
            continue
        adj_full = read_bool_dag(dag_path)
        adj, original, scope = graph_scope_matrices(dataset, family, adj_full, original_full)

        edge_count = int(adj.sum())
        original_edge_count = int(original.sum())
        diff = adj != original
        preserved_edges = int((adj & original).sum())
        edge_overlap = preserved_edges / original_edge_count if original_edge_count else np.nan
        rewire_fraction = 1.0 - edge_overlap if pd.notna(edge_overlap) else np.nan
        rows.append({
            'dataset': dataset,
            'family': family,
            'variant': variant,
            'graph_scope': scope,
            'dag_path': str(dag_path),
            'edge_count': edge_count,
            'original_edge_count': original_edge_count,
            'hamming_distance': int(diff.sum()),
            'graph_distance': float(diff.sum() / diff.size) if diff.size else np.nan,
            'preserved_edges': preserved_edges,
            'edge_overlap': edge_overlap,
            'rewire_fraction': rewire_fraction,
            'rewire_percent': 100.0 * rewire_fraction if pd.notna(rewire_fraction) else np.nan,
            'added_edges': int((adj & ~original).sum()),
            'deleted_edges': int((~adj & original).sum()),
        })
    return pd.DataFrame(rows)


def normalize_group_value(value):
    if pd.isna(value):
        return 'all'
    text = str(value).strip().lower()
    if text in {'false', '0', '0.0'}:
        return 'false'
    if text in {'true', '1', '1.0'}:
        return 'true'
    return text.replace(' ', '_')


def intervention_summary(interventions_df):
    if interventions_df.empty:
        return pd.DataFrame()
    acc_col = intervention_accuracy_column(interventions_df)
    if acc_col is None or 'num_interventions' not in interventions_df.columns:
        return pd.DataFrame()

    rows = []
    group_cols = ['dataset', 'family', 'variant']
    if 'group_interventions' in interventions_df.columns:
        grouped = interventions_df.groupby(group_cols + ['group_interventions'], dropna=False)
    else:
        grouped = [((*key, np.nan), group) for key, group in interventions_df.groupby(group_cols)]

    for key, group in grouped:
        dataset, family, variant, group_value = key
        curve = group.dropna(subset=['num_interventions', acc_col]).sort_values('num_interventions')
        if curve.empty:
            continue
        x = curve['num_interventions'].to_numpy(dtype=float)
        y = curve[acc_col].to_numpy(dtype=float)
        x_norm = x / x.max() if x.max() > 0 else x
        suffix = normalize_group_value(group_value)
        rows.append({
            'dataset': dataset,
            'family': family,
            'variant': variant,
            f'intervention_auc_{suffix}': float(np.trapz(y, x_norm)) if len(x_norm) > 1 else float(y[0]),
            f'intervention_start_accuracy_{suffix}': float(y[0]),
            f'intervention_final_accuracy_{suffix}': float(y[-1]),
            f'intervention_max_accuracy_{suffix}': float(np.max(y)),
            f'intervention_points_{suffix}': int(len(curve)),
        })

    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame(rows)
    value_cols = [c for c in raw.columns if c not in {'dataset', 'family', 'variant'}]
    return raw.groupby(['dataset', 'family', 'variant'], as_index=False)[value_cols].first()


def add_reference_deltas(summary):
    if summary.empty:
        return summary
    out = summary.copy()
    delta_specs = [
        ('test_task_accuracy', 'delta_pred_task_to_original'),
        ('test_concept_accuracy', 'delta_pred_concept_to_original'),
        ('intervention_auc_false', 'delta_int_auc_false_to_original'),
        ('intervention_auc_true', 'delta_int_auc_true_to_original'),
    ]
    for _, group_idx in out.groupby(['dataset', 'family']).groups.items():
        group = out.loc[group_idx]
        original = group[group['variant'] == 'original_CREAM']
        if original.empty and 'is_original_reference' in group.columns:
            original = group[group['is_original_reference'].fillna(False).astype(bool)]
        if original.empty:
            continue
        original = original.iloc[0]
        for src, dst in delta_specs:
            if src in out.columns:
                out.loc[group_idx, dst] = (group[src] - original.get(src, np.nan)).abs()

    # One default intervention delta for plotting/tables: prefer individual-concept interventions.
    if 'delta_int_auc_false_to_original' in out.columns and out['delta_int_auc_false_to_original'].notna().any():
        out['delta_int_to_original'] = out['delta_int_auc_false_to_original']
    elif 'delta_int_auc_true_to_original' in out.columns:
        out['delta_int_to_original'] = out['delta_int_auc_true_to_original']
    else:
        out['delta_int_to_original'] = np.nan
    return out


def make_family_summary(metrics_df, interventions_df):
    if metrics_df.empty:
        return pd.DataFrame()
    summary = metrics_df.copy()

    # Always expose leakage columns. Standard simple_main CREAM runs may not write
    # concept_leakage; in that case the column stays NaN instead of disappearing.
    if 'concept_leakage' not in summary.columns:
        summary['concept_leakage'] = np.nan
    if {'test_task_accuracy', 'c2y_baseline_accuracy'}.issubset(summary.columns):
        fallback_leakage = (summary['test_task_accuracy'] - summary['c2y_baseline_accuracy']).clip(lower=0)
        summary['concept_leakage'] = summary['concept_leakage'].fillna(fallback_leakage)
    if 'data_leakage' not in summary.columns:
        summary['data_leakage'] = summary['concept_leakage']

    graph_props = graph_property_rows_for_results(summary)
    if not graph_props.empty:
        summary = summary.merge(graph_props, on=['dataset', 'family', 'variant'], how='left')

    int_summary = intervention_summary(interventions_df)
    if not int_summary.empty:
        summary = summary.merge(int_summary, on=['dataset', 'family', 'variant'], how='left')
    summary = add_reference_deltas(summary)
    return summary


def make_gap_pairs(summary, family):
    fam = summary[summary['family'] == family].copy() if not summary.empty else pd.DataFrame()
    if fam.empty:
        return pd.DataFrame()

    rows = []
    for dataset, group in fam.groupby('dataset'):
        records = group.to_dict('records')
        for i in range(len(records)):
            for j in range(i + 1, len(records)):
                a, b = records[i], records[j]
                row = {
                    'dataset': dataset,
                    'family': family,
                    'variant_i': a.get('variant'),
                    'variant_j': b.get('variant'),
                    'delta_pred_task': abs(a.get('test_task_accuracy', np.nan) - b.get('test_task_accuracy', np.nan)),
                    'delta_pred_concept': abs(a.get('test_concept_accuracy', np.nan) - b.get('test_concept_accuracy', np.nan)),
                    'delta_int_auc_false': abs(a.get('intervention_auc_false', np.nan) - b.get('intervention_auc_false', np.nan)),
                    'delta_int_auc_true': abs(a.get('intervention_auc_true', np.nan) - b.get('intervention_auc_true', np.nan)),
                    'delta_rewire_percent': abs(a.get('rewire_percent', np.nan) - b.get('rewire_percent', np.nan)),
                    'delta_hamming_distance': abs(a.get('hamming_distance', np.nan) - b.get('hamming_distance', np.nan)),
                }
                row['delta_int'] = row['delta_int_auc_false'] if pd.notna(row['delta_int_auc_false']) else row['delta_int_auc_true']
                rows.append(row)
    return pd.DataFrame(rows)



def enrich_interventions_for_export(interventions_df, summary_df):
    if interventions_df.empty:
        return interventions_df.copy()
    enriched = interventions_df.copy()
    if summary_df.empty:
        return enriched

    export_cols = [
        'dataset', 'family', 'variant',
        'graph_scope', 'edge_count', 'original_edge_count',
        'hamming_distance', 'graph_distance', 'preserved_edges',
        'edge_overlap', 'rewire_fraction', 'rewire_percent',
        'test_task_accuracy', 'test_concept_accuracy', 'concept_leakage', 'data_leakage',
        'intervention_auc_false', 'intervention_auc_true',
        'intervention_final_accuracy_false', 'intervention_final_accuracy_true',
        'delta_pred_task_to_original', 'delta_pred_concept_to_original',
        'delta_int_auc_false_to_original', 'delta_int_auc_true_to_original', 'delta_int_to_original',
    ]
    summary_cols = available(summary_df.columns, export_cols)
    extra = summary_df[summary_cols].drop_duplicates(['dataset', 'family', 'variant'])

    merge_extra_cols = [c for c in extra.columns if c not in enriched.columns or c in {'dataset', 'family', 'variant'}]
    enriched = enriched.merge(extra[merge_extra_cols], on=['dataset', 'family', 'variant'], how='left')

    # Put identity and graph descriptors first so each intervention curve is self-describing.
    first_cols = available(enriched.columns, [
        'dataset', 'family', 'variant', 'is_original_reference',
        'graph_scope', 'edge_count', 'original_edge_count',
        'hamming_distance', 'graph_distance', 'preserved_edges',
        'edge_overlap', 'rewire_fraction', 'rewire_percent',
        'group_interventions', 'num_interventions',
    ])
    remaining_cols = [c for c in enriched.columns if c not in first_cols]
    return enriched[first_cols + remaining_cols]


def save_family_exports(family, family_table, interventions_df, gap_pairs):
    exported = []
    family_dir = EXPORT_DIR / family
    family_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = family_dir / f'{family}_family_results.csv'
    family_table.to_csv(metrics_path, index=False)
    exported.append(metrics_path)

    if not gap_pairs.empty:
        gap_path = family_dir / f'{family}_observational_interventional_gap_pairs.csv'
        gap_pairs.to_csv(gap_path, index=False)
        exported.append(gap_path)

    fam_interventions = interventions_df[interventions_df['family'] == family].copy() if not interventions_df.empty else pd.DataFrame()
    if not fam_interventions.empty:
        fam_interventions = enrich_interventions_for_export(fam_interventions, family_table)
        all_path = family_dir / f'{family}_interventions_all_datasets.csv'
        fam_interventions.to_csv(all_path, index=False)
        exported.append(all_path)
        for dataset, dataset_df in fam_interventions.groupby('dataset'):
            dataset_path = family_dir / f'{family}_{dataset}_interventions.csv'
            dataset_df.to_csv(dataset_path, index=False)
            exported.append(dataset_path)
    return exported


def plot_gap_pairs(gap_pairs, family):
    if gap_pairs.empty:
        print('No pairwise observational/interventional gap rows yet.')
        return
    plot_df = gap_pairs.dropna(subset=['delta_pred_task', 'delta_int'])
    if plot_df.empty:
        print('No plottable pairwise gap rows yet.')
        return
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    for dataset, dataset_df in plot_df.groupby('dataset'):
        ax.scatter(dataset_df['delta_pred_task'], dataset_df['delta_int'], s=55, label=dataset, alpha=0.85)
    ax.set_xlabel(r'$\Delta_{pred}$ task accuracy')
    ax.set_ylabel(r'$\Delta_{int}$ intervention AUC')
    ax.set_title(f'{family}: observational--interventional gap')
    ax.grid(alpha=0.3)
    ax.legend(title='Dataset')
    plt.tight_layout()
    plt.show()


def display_family_results(family, metrics_df, interventions_df):
    print(f"\n=== {family}: {FAMILIES[family]} ===")
    summary = make_family_summary(metrics_df, interventions_df)
    fam_metrics = summary[summary['family'] == family].copy() if not summary.empty else pd.DataFrame()
    if fam_metrics.empty:
        print('No metrics found yet for this family.')
        return fam_metrics

    cols = available(fam_metrics.columns, KEY_METRICS)
    sort_cols = available(fam_metrics.columns, ['dataset', 'is_original_reference', 'variant'])
    table = fam_metrics[cols].sort_values(sort_cols, ascending=[True, False, True] if len(sort_cols) == 3 else True)
    display(table)

    gap_pairs = make_gap_pairs(summary, family)
    exported = save_family_exports(family, table, interventions_df, gap_pairs)
    print('Saved family CSV exports:')
    for path in exported:
        print('  ', path)

    if not gap_pairs.empty:
        display_cols = available(
            gap_pairs.columns,
            ['dataset', 'family', 'variant_i', 'variant_j', 'delta_pred_task', 'delta_pred_concept', 'delta_int_auc_false', 'delta_int_auc_true', 'delta_int', 'delta_rewire_percent', 'delta_hamming_distance'],
        )
        display(gap_pairs[display_cols].sort_values(['dataset', 'delta_pred_task', 'delta_int'], ascending=[True, True, False]).head(40))
        plot_gap_pairs(gap_pairs, family)

    fam_interventions = interventions_df[interventions_df['family'] == family].copy() if not interventions_df.empty else pd.DataFrame()
    if fam_interventions.empty:
        print('No intervention curves found yet for this family.')
        return table
    acc_col = intervention_accuracy_column(fam_interventions)
    if acc_col is None or 'num_interventions' not in fam_interventions.columns:
        print('Intervention CSVs found, but no plottable accuracy/num_interventions columns.')
        display(fam_interventions.head())
        return table

    for dataset, dataset_df in fam_interventions.groupby('dataset'):
        group_values = [None]
        if 'group_interventions' in dataset_df.columns:
            group_values = sorted(dataset_df['group_interventions'].dropna().unique())
        for group_value in group_values:
            df = dataset_df if group_value is None else dataset_df[dataset_df['group_interventions'] == group_value]
            if df.empty:
                continue
            fig, ax = plt.subplots(figsize=(8.5, 5))
            for variant, variant_df in df.groupby('variant'):
                variant_df = variant_df.dropna(subset=['num_interventions', acc_col]).sort_values('num_interventions')
                if variant_df.empty:
                    continue
                is_original = bool(variant_df['is_original_reference'].iloc[0]) if 'is_original_reference' in variant_df.columns else False
                linewidth = 3.0 if is_original else 1.8
                linestyle = '--' if is_original else '-'
                ax.plot(
                    variant_df['num_interventions'],
                    variant_df[acc_col],
                    marker='o',
                    linewidth=linewidth,
                    linestyle=linestyle,
                    label=variant,
                )
            suffix = '' if group_value is None else f' | group_interventions={group_value}'
            ax.set_title(f'{dataset}: {family} intervention curve{suffix}')
            ax.set_xlabel('Number of interventions')
            ax.set_ylabel(acc_col)
            ax.grid(True, alpha=0.3)
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
    return table


## Load Results


In [ ]:

metrics = load_result_metrics()
interventions = load_result_interventions()
summary_metrics = make_family_summary(metrics, interventions)

print(f'Loaded metric rows: {len(metrics)}')
print(f'Loaded intervention rows: {len(interventions)}')
print(f'Prepared summary rows: {len(summary_metrics)}')
print(f'Export directory: {EXPORT_DIR}')

if not metrics.empty:
    metrics_out = PROJECT_ROOT / 'notebook/topology_density_c2c_results_metrics_raw.csv'
    metrics.to_csv(metrics_out, index=False)
    print(f'Saved raw metrics table to {metrics_out}')

if not summary_metrics.empty:
    summary_out = PROJECT_ROOT / 'notebook/topology_density_c2c_results_metrics.csv'
    summary_metrics.to_csv(summary_out, index=False)
    print(f'Saved enriched metrics table to {summary_out}')

if not interventions.empty:
    interventions_out = PROJECT_ROOT / 'notebook/topology_density_c2c_results_interventions.csv'
    enriched_interventions = enrich_interventions_for_export(interventions, summary_metrics)
    enriched_interventions.to_csv(interventions_out, index=False)
    print(f'Saved enriched intervention table to {interventions_out}')


## All Result Metrics


In [ ]:
if summary_metrics.empty:
    print('No metrics found yet. Run the Condor jobs first, then rerun this notebook.')
else:
    cols = available(summary_metrics.columns, KEY_METRICS)
    display(summary_metrics[cols].sort_values(['family', 'dataset', 'is_original_reference', 'variant'], ascending=[True, True, False, True]))


## Full-DAG Topology

Family key: `topology_random_full`


In [ ]:
display_family_results('topology_random_full', metrics, interventions)


## Full-DAG Density

Family key: `density_random_full`


In [ ]:
display_family_results('density_random_full', metrics, interventions)


## C--C Fixed-Count Topology

Family key: `concept_concept_fixcount_topology`


In [ ]:
display_family_results('concept_concept_fixcount_topology', metrics, interventions)
